In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity

import json

In [2]:
from sentence_transformers import SentenceTransformer

model =SentenceTransformer("all-MiniLM-L6-v2",device='cuda')


/home/vighnesh/projects/coursera/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████| 103/103 [00:00<00:00, 1658.83it/s]


In [3]:
movies_df = pd.read_csv('../dataset/processed/movies_df_clean.csv')

In [4]:
movies_df['overview_clean'].isna().sum()

np.int64(0)

In [5]:
' '+', '.join([])

' '

In [6]:
movies_df.isna().sum()

movie_id                     0
budget                       0
homepage                  3086
original_title               0
popularity                   0
production_companies         0
production_countries         0
release_date                 1
revenue                      0
runtime                      2
spoken_languages             0
tagline                      0
title                        0
vote_average                 0
vote_count                   0
overview_clean               0
genres_clean                 0
cast_clean                   0
crew_clean                   0
keywords_clean               0
original_language_full       0
dtype: int64

In [7]:
def combine_all(x):
    text = f'''Genres : {x['genres_clean']}.
Keywords : {x['keywords_clean']}.
{x['crew_clean']}.
Cast : {x['cast_clean']}.
Overview :  {x['overview_clean'] }
'''.strip()

    return text
# Tagline is "{x['tagline'][:-1]}".

In [8]:
movies_df['combined_text'] =  movies_df.apply(combine_all ,axis=1)

In [9]:
print(movies_df['combined_text'][0])

Genres :  Fantasy, Science Fiction, Action, Adventure.
Keywords :  space war, future, love affair, culture clash, romance, space, space travel, soldier, space colony, power relations, mind and soul, society, battle, tribe, marine, futuristic, alien, 3d, cgi, anti war, alien planet.
Director: James Cameron.
Cast :  Sam Worthington, Zoe Saldana, Sigourney Weaver, Stephen Lang.
Overview :  In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.


### Build Model !!

In [10]:
embedding = model.encode(
    inputs=movies_df['combined_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches: 100%|███████████████████████████████████████████████████████████| 75/75 [00:04<00:00, 18.53it/s]


In [11]:
embedding.shape

(4795, 384)

In [12]:
# verify the embedding
movie_idx = movies_df[movies_df['title']=='Interstellar'].index[0]


In [13]:
# compute Similarities 
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [embedding[movie_idx]],embedding
)[0]

In [14]:
top_indicies = similarities.argsort()[::-1][:11]

In [15]:
for idx in top_indicies:
    print(movies_df.loc[idx,'title'], similarities[idx])

Interstellar 1.0000001
Prometheus 0.65075994
The Black Hole 0.6291444
Seeking a Friend for the End of the World 0.6204966
The Martian 0.61358416
Gattaca 0.6093078
Red Planet 0.6090753
Lost in Space 0.60705566
Meet Dave 0.60654265
Mission to Mars 0.60278755
Close Encounters of the Third Kind 0.6006805


### 3. composite model

In [16]:
genre_emb = model.encode(inputs=movies_df['genres_clean'].tolist(),show_progress_bar=True,
                         batch_size=64, convert_to_numpy=True)
overview_emb = model.encode(inputs=movies_df['overview_clean'].tolist(),show_progress_bar=True,
                            batch_size=64,convert_to_numpy=True)
keyword_emb = model.encode(inputs=movies_df['keywords_clean'].tolist(),show_progress_bar=True,
                           batch_size=64, convert_to_numpy=True)
director_emb = model.encode(inputs=movies_df['crew_clean'].tolist(),show_progress_bar=True,
                            batch_size=64, convert_to_numpy=True)

Batches: 100%|██████████████████████████████████████████████████████████| 75/75 [00:00<00:00, 139.21it/s]


In [23]:
genre_emb.

array([[ 0.03145788, -0.05792385,  0.00292396, ..., -0.04490043,
        -0.05579026,  0.04191204],
       [ 0.01862613, -0.01425429, -0.01942158, ..., -0.05856192,
        -0.06296689,  0.01208922],
       [ 0.00497166,  0.02522605, -0.05202925, ..., -0.01200447,
        -0.00608318, -0.06969757],
       ...,
       [-0.01344521, -0.08258423,  0.02264843, ...,  0.01607076,
         0.0556645 , -0.02600691],
       [-0.11883835,  0.04829865, -0.00254809, ...,  0.1264094 ,
         0.04654898, -0.01571719],
       [-0.08694077,  0.02698923, -0.03981005, ..., -0.00016568,
         0.06227834, -0.03243548]], shape=(4795, 384), dtype=float32)

In [17]:
# compute Similarities 

genre_similarities = cosine_similarity(
    [genre_emb[movie_idx]],genre_emb
)[0]

overview_similarities = cosine_similarity(
    [overview_emb[movie_idx]],overview_emb
)[0]

keyword_similarities = cosine_similarity(
    [keyword_emb[movie_idx]],keyword_emb
)[0]

director_similarities = cosine_similarity(
    [director_emb[movie_idx]],director_emb
)[0]

In [18]:
final_emb = (
    0.30 * genre_emb +
    0.30 * keyword_emb +
    0.35 * overview_emb +
    0.05 * director_emb
)


In [35]:
EMBEDDING_DIR_PATH = os.path.join('..','artifact','embeddings')
def save_embedding(file_name,emdd):
    try:
        emdd_path =os.path.join(EMBEDDING_DIR_PATH,file_name)
        np.save(emdd_path,arr=emdd)
        print(f'{file_name} saved successfull!')
    except Exception as e:
        print(f"Error process stop : {e}")
        

    
    

# save final embedded and other embedded arrays
save_embedding('final_embeddings.npy',final_emb)
save_embedding('genre_embeddings.npy',genre_emb)
save_embedding('overview_embeddings.npy',overview_emb)
save_embedding('keyword_embeddings.npy',keyword_emb)
save_embedding('director_embeddings.npy',director_emb)

final_embeddings.npy saved successfull!
genre_embeddings.npy saved successfull!
overview_embeddings.npy saved successfull!
keyword_embeddings.npy saved successfull!
director_embeddings.npy saved successfull!


### Demo of process

In [19]:
similarities = cosine_similarity(
    [final_emb[movie_idx]],final_emb
)[0]

In [20]:
top_indicies = similarities.argsort()[::-1][:11]

In [21]:
for idx in top_indicies:
    print(movies_df.loc[idx,'title'], similarities[idx])

Interstellar 1.0
Prometheus 0.801999
The Hitchhiker's Guide to the Galaxy 0.75001806
Meet Dave 0.7439928
E.T. the Extra-Terrestrial 0.7398513
A.I. Artificial Intelligence 0.73749286
The Martian 0.73300195
The Inhabited Island 0.7320565
Gattaca 0.7306923
Home 0.7277653
Treasure Planet 0.7273561


In [50]:
def get_similarity_explaination(movie_idx,remm_idx):
    gen_sim = cosine_similarity(genre_emb[movie_idx].reshape(1,-1),genre_emb[remm_idx].reshape(1,-1))[0,0]

    overview_sim = cosine_similarity(overview_emb[movie_idx].reshape(1,-1),overview_emb[remm_idx].reshape(1,-1))[0,0]

    keyword_sim = cosine_similarity(keyword_emb[movie_idx].reshape(1,-1),keyword_emb[remm_idx].reshape(1,-1))[0,0]

    director_sim = cosine_similarity(director_emb[movie_idx].reshape(1,-1),director_emb[remm_idx].reshape(1,-1))[0,0]

    final_score =(gen_sim*30 + overview_sim*35 + keyword_sim  *30 + director_sim *0.05)

    return {'genre_similarity' : gen_sim, 
            "keyword_similarity": keyword_sim,
        "overview_similarity": overview_sim,
        "director_similarity": director_sim,
        "final_score": final_score
           }
    

In [64]:
get_similarity_explaination(95,220)

{'genre_similarity': np.float32(0.8292588),
 'keyword_similarity': np.float32(0.5708525),
 'overview_similarity': np.float32(0.5993278),
 'director_similarity': np.float32(0.67599505),
 'final_score': np.float32(63.01361)}

In [24]:
overview_emb.shape

(4795, 384)

In [63]:
movies_df[movies_df['title'] == 'Prometheus']

,movie_id,budget,homepage,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,...,tagline,title,vote_average,vote_count,overview_clean,genres_clean,cast_clean,crew_clean,keywords_clean,original_language_full
220,70981,130000000,http://www.projectprometheus.com/,Prometheus,68.889395,"[{""name"": ""Twentieth Century Fox Film Corporat...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2012-05-30,403170142,124.0,...,The Search for Our Beginning Could Lead to Our...,Prometheus,6.3,5080,A team of explorers discover a clue to the ori...,"Science Fiction, Mystery, Adventure","Noomi Rapace, Michael Fassbender, Guy Pearce,...",Director: Ridley Scott,"alien, stasis, creation, spin off, cave drawi...",English


In [45]:
cosine_similarity(genre_emb[1].reshape(1,-1),genre_emb[6].reshape(1,-1))[0][0]

np.float32(0.41295248)